### Preprocessing for Better Retrieval (table -> markdown)

In [38]:
# %pip install --upgrade -q docx2txt langchain-community

In [39]:
# %pip install -qU langchain-text-splitters

In [40]:
# 1,2 : read&load document -> split document

from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader("./tax_with_markdown.docx")
# split document
document_list = loader.load_and_split(text_splitter=text_splitter)

In [41]:
len(document_list)

194

In [42]:
# pull text embedding model (text to vector)
# !ollama pull nomic-embed-text

In [43]:
# 3: Embedding
from langchain_ollama import OllamaEmbeddings

embedding = OllamaEmbeddings(model="nomic-embed-text")

# dimension 확인용 test
# test = embedding.embed_query("test")
# print(len(test))  # output을 pinecone index의 dimensions에 넣어

In [44]:
# %pip install --upgrade --quiet \
#     langchain-pinecone \
#     langchain-ollama \
#     langchain

In [45]:
from dotenv import load_dotenv
import os
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

# pinecone 연결
load_dotenv()
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)

# pinecone index에 document_list를 embedding
database = PineconeVectorStore.from_documents(
    document_list, embedding, index_name="tax-markdown-index"
)

In [46]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"

In [47]:
# 5: Pass retrieved documents along with the question to the LLM
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3:latest")

In [48]:
# RetrievalQA Chain
# %pip install -U langchain langchainhub -q

In [49]:
from langchain_classic import hub

# get prompt from hub
prompt = hub.pull("rlm/rag-prompt")

In [ ]:
# docx에서 query에 해당하는 부분을 image -> table로 변경했음에도 관련 문서로 인식 실패
retriever = database.as_retriever(search_kwargs={"k": 10})
retriever.invoke(query)

In [51]:
# Build QA Chain
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm, retriever=retriever, chain_type_kwargs={"prompt": prompt}
)
ai_message = qa_chain.invoke({"query": query})

In [52]:
ai_message

{'query': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'result': "I don't know the answer to this question because the given context does not provide information about the specific income of the individual with an annual salary of 5,000 million won. The provided laws and regulations do not seem to be related to individual income tax calculations. To determine the income tax for a person with an annual salary of 5,000 million won, you would need more detailed information about their income, deductions, and other relevant factors."}

In [ ]:
# 세율표가 어떤 chunk에 들어있는지 확인
for i, doc in enumerate(document_list):
    if "과세표준" in doc.page_content and "세율" in doc.page_content:
        print(f"=== Chunk {i} ===")
        print(doc.page_content[:300])
        print()

In [ ]:
# 문서 용어에 맞춘 쿼리로 테스트 - 이걸로 찾으면 embedding 모델의 용어 매칭 문제
query2 = "종합소득 과세표준 세율"
retriever.invoke(query2)

### Findings
- With `k=10`, Article 55 (tax rate table) is now included in the retrieved documents
  - `query` ("연봉 5천만원 소득세") → Article 55 ranked **5th** — weak semantic matching between "연봉/소득세" and "과세표준/세율"
  - `query2` ("종합소득 과세표준 세율") → Article 55 ranked **3rd** — direct keyword match improves ranking
- Despite successful retrieval, `llama3` still fails to generate a correct answer — it struggles to extract the relevant section from 10 documents and perform the tax calculation
- This is likely a **LLM capability gap** compared to `gpt-4o`, which handles document comprehension and numeric reasoning much better